# Face capturing

In [ ]:
# import dependencies
from IPython.display import display, Javascript, Image, clear_output
from google.colab.output import eval_js
from base64 import b64decode, b64encode
import cv2
import numpy as np
import PIL
import io
import os
import time

# Function to convert JavaScript object into OpenCV image
def js_to_image(js_reply):
    image_bytes = b64decode(js_reply.split(',')[1])
    jpg_as_np = np.frombuffer(image_bytes, dtype=np.uint8)
    img = cv2.imdecode(jpg_as_np, flags=1)
    return img

# Function to convert OpenCV Rectangle bounding box image into base64 byte string for overlay
def bbox_to_bytes(bbox_array):
    bbox_PIL = PIL.Image.fromarray(bbox_array, 'RGBA')
    iobuf = io.BytesIO()
    bbox_PIL.save(iobuf, format='png')
    bbox_bytes = 'data:image/png;base64,{}'.format((str(b64encode(iobuf.getvalue()), 'utf-8')))
    return bbox_bytes

# Initialize Haar Cascade face detection model
face_cascade = cv2.CascadeClassifier(cv2.samples.findFile(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'))

# JavaScript to create the video stream using the webcam
def video_stream():
    js = Javascript('''
        var video;
        var div = null;
        var stream;
        var captureCanvas;
        var imgElement;
        var labelElement;

        var pendingResolve = null;
        var shutdown = false;

        function removeDom() {
           stream.getVideoTracks()[0].stop();
           video.remove();
           div.remove();
           video = null;
           div = null;
           stream = null;
           imgElement = null;
           captureCanvas = null;
           labelElement = null;
        }

        function onAnimationFrame() {
          if (!shutdown) {
            window.requestAnimationFrame(onAnimationFrame);
          }
          if (pendingResolve) {
            var result = "";
            if (!shutdown) {
              captureCanvas.getContext('2d').drawImage(video, 0, 0, 640, 480);
              result = captureCanvas.toDataURL('image/jpeg', 0.8)
            }
            var lp = pendingResolve;
            pendingResolve = null;
            lp(result);
          }
        }

        async function createDom() {
          if (div !== null) {
            return stream;
          }

          div = document.createElement('div');
          div.style.border = '2px solid black';
          div.style.padding = '3px';
          div.style.width = '100%';
          div.style.maxWidth = '600px';
          document.body.appendChild(div);

          const modelOut = document.createElement('div');
          modelOut.innerHTML = "<span>Status:</span>";
          labelElement = document.createElement('span');
          labelElement.innerText = 'No data';
          labelElement.style.fontWeight = 'bold';
          modelOut.appendChild(labelElement);
          div.appendChild(modelOut);

          video = document.createElement('video');
          video.style.display = 'block';
          video.width = div.clientWidth - 6;
          video.setAttribute('playsinline', '');
          video.onclick = () => { shutdown = true; };
          stream = await navigator.mediaDevices.getUserMedia(
              {video: { facingMode: "environment"}});
          div.appendChild(video);

          imgElement = document.createElement('img');
          imgElement.style.position = 'absolute';
          imgElement.style.zIndex = 1;
          imgElement.onclick = () => { shutdown = true; };
          div.appendChild(imgElement);

          const instruction = document.createElement('div');
          instruction.innerHTML =
              '<span style="color: white; font-weight: bold;">' +
              'When finished, click here or on the video to stop this demo</span>';
          div.appendChild(instruction);
          instruction.onclick = () => { shutdown = true; };

          video.srcObject = stream;
          await video.play();

          captureCanvas = document.createElement('canvas');
          captureCanvas.width = 640;
          captureCanvas.height = 480;
          window.requestAnimationFrame(onAnimationFrame);

          return stream;
        }
        async function stream_frame(label, imgData) {
          if (shutdown) {
            removeDom();
            shutdown = false;
            return '';
          }

          var preCreate = Date.now();
          stream = await createDom();

          var preShow = Date.now();
          if (label != "") {
            labelElement.innerHTML = label;
          }

          if (imgData != "") {
            var videoRect = video.getClientRects()[0];
            imgElement.style.top = videoRect.top + "px";
            imgElement.style.left = videoRect.left + "px";
            imgElement.style.width = videoRect.width + "px";
            imgElement.style.height = videoRect.height + "px";
            imgElement.src = imgData;
          }

          var preCapture = Date.now();
          var result = await new Promise(function(resolve, reject) {
            pendingResolve = resolve;
          });
          shutdown = false;

          return {'create': preShow - preCreate,
                  'show': preCapture - preShow,
                  'capture': Date.now() - preCapture,
                  'img': result};
        }
        ''')

    display(js)

def video_frame(label, bbox):
    data = eval_js('stream_frame("{}", "{}")'.format(label, bbox))
    return data

# Start streaming video from webcam
video_stream()

# Set up user folder for saving images
person_name = input("Enter the person's name: ")
save_path = f"/content/drive/MyDrive/face_rec/{person_name}"
os.makedirs(save_path, exist_ok=True)

# Define variables for capturing process
label_html = 'Capturing...'
bbox = ''
count = 0
max_images = 15
positions = {"center": "c", "left": "l", "right": "r"}

# Define the central rectangle region in the screen
rect_x, rect_y, rect_w, rect_h = 220, 140, 200, 200

# Capture images for each position with designated naming convention
for pos, pos_prefix in positions.items():
    print(f"Position your face in the center and turn {pos}.")
    for countdown in range(3, 0, -1):
        print(f"Starting in {countdown} seconds...")
        time.sleep(1)

    count = 0
    while count < max_images:
        js_reply = video_frame(f"Position: {pos} | Face in Box", bbox)
        if not js_reply:
            break

        img = js_to_image(js_reply["img"])
        bbox_array = np.zeros([480, 640, 4], dtype=np.uint8)

        gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
        faces = face_cascade.detectMultiScale(gray)

        # Draw central cyan rectangle
        bbox_array = cv2.rectangle(bbox_array, (rect_x, rect_y), (rect_x + rect_w, rect_y + rect_h), (0, 255, 255), 2)

        for (x, y, w, h) in faces:
            if (rect_x < x < rect_x + rect_w - w) and (rect_y < y < rect_y + rect_h - h):
                count += 1
                image_name = f"{person_name}_{pos_prefix}{count}.jpg"  # Name image as per person, position, and count (e.g., Name_c1)
                face_img = img[y:y+h, x:x+w]
                cv2.imwrite(f"{save_path}/{image_name}", face_img)
                print(f"Captured image {image_name} for {pos} position")
                break

        bbox_array[:, :, 3] = (bbox_array.max(axis=2) > 0).astype(int) * 255
        bbox_bytes = bbox_to_bytes(bbox_array)
        bbox = bbox_bytes

# Display and list captured images for review
captured_images = os.listdir(save_path)
print("Captured Images:")
for image_name in captured_images:
    image_path = os.path.join(save_path, image_name)
    display(Image(filename=image_path))  # Display image inline
    print(image_name)

# Ask user to input image IDs to delete without the name prefix (e.g., c5, l2, r3)
delete_input = input("Enter image IDs to delete (e.g., c5, l2, r3): ")
delete_list = delete_input.split(',')

# Delete specified images
for img_id in delete_list:
    img_filename = f"{person_name}_{img_id.strip()}.jpg"  # Add name prefix and .jpg extension
    img_path = os.path.join(save_path, img_filename)
    if os.path.exists(img_path):
        os.remove(img_path)
        print(f"Deleted {img_filename}")
    else:
        print(f"Image {img_filename} not found")




In [ ]:
load_dataset_to_excel(save_path)

In [ ]:
# Load MobileNetV2 model for feature extraction
base_model = MobileNetV2(weights='imagenet', include_top=False, pooling='avg', input_shape=(224, 224, 3))
model = Model(inputs=base_model.input, outputs=base_model.output)
def load_dataset_to_excel(dataset_path, output_file="embeddings.xlsx"):
    data = []  # List to store image names, labels, and embeddings

    for person_name in os.listdir(dataset_path):
        person_folder = os.path.join(dataset_path, person_name)
        if os.path.isdir(person_folder):
            for image_name in os.listdir(person_folder):
                image_path = os.path.join(person_folder, image_name)
                img = cv2.imread(image_path)

                # Resize and normalize image
                img = cv2.resize(img, (224, 224))
                img = img.astype('float32') / 255.0
                img = np.expand_dims(img, axis=0)

                # Get the embedding from the MobileNetV2 model
                embedding = model.predict(img)[0]

                # Add data to the list: [image_name, person_name, embedding]
                data.append([image_name, person_name, embedding])

    # Convert the data to a pandas DataFrame
    df = pd.DataFrame(data, columns=["Image Name", "Label", "Embedding"])

    # Split embeddings into separate columns for easier viewing in Excel
    embedding_df = pd.DataFrame(df["Embedding"].tolist())
    df = df.drop(columns=["Embedding"]).join(embedding_df)

    # Check if the output file already exists
    if os.path.exists(output_file):
        # Load existing data and append the new data
        existing_df = pd.read_excel(output_file)
        df = pd.concat([existing_df, df], ignore_index=True)

    # Save the DataFrame to the Excel file
    df.to_excel(output_file, index=False)
    print(f"Embeddings saved to {output_file}")

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


# Raw code without ui


In [ ]:
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode, b64encode
import cv2
import numpy as np
import PIL
import io
import os
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.applications import MobileNetV2
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

In [ ]:
# JavaScript to properly create our live video stream using our webcam as input
def video_stream():
  js = Javascript('''
    var video;
    var div = null;
    var stream;
    var captureCanvas;
    var imgElement;
    var labelElement;

    var pendingResolve = null;
    var shutdown = false;

    function removeDom() {
       stream.getVideoTracks()[0].stop();
       video.remove();
       div.remove();
       video = null;
       div = null;
       stream = null;
       imgElement = null;
       captureCanvas = null;
       labelElement = null;
    }

    function onAnimationFrame() {
      if (!shutdown) {
        window.requestAnimationFrame(onAnimationFrame);
      }
      if (pendingResolve) {
        var result = "";
        if (!shutdown) {
          captureCanvas.getContext('2d').drawImage(video, 0, 0, 640, 480);
          result = captureCanvas.toDataURL('image/jpeg', 0.8)
        }
        var lp = pendingResolve;
        pendingResolve = null;
        lp(result);
      }
    }

    async function createDom() {
      if (div !== null) {
        return stream;
      }

      div = document.createElement('div');
      div.style.border = '2px solid black';
      div.style.padding = '3px';
      div.style.width = '100%';
      div.style.maxWidth = '600px';
      document.body.appendChild(div);

      const modelOut = document.createElement('div');
      modelOut.innerHTML = "<span>Status:</span>";
      labelElement = document.createElement('span');
      labelElement.innerText = 'No data';
      labelElement.style.fontWeight = 'bold';
      modelOut.appendChild(labelElement);
      div.appendChild(modelOut);

      video = document.createElement('video');
      video.style.display = 'block';
      video.width = div.clientWidth - 6;
      video.setAttribute('playsinline', '');
      video.onclick = () => { shutdown = true; };
      stream = await navigator.mediaDevices.getUserMedia(
          {video: { facingMode: "environment"}});
      div.appendChild(video);

      imgElement = document.createElement('img');
      imgElement.style.position = 'absolute';
      imgElement.style.zIndex = 1;
      imgElement.onclick = () => { shutdown = true; };
      div.appendChild(imgElement);

      const instruction = document.createElement('div');
      instruction.innerHTML =
          '<span style="color: white; font-weight: bold;">' +
          'When finished, click here or on the video to stop this demo</span>';
      div.appendChild(instruction);
      instruction.onclick = () => { shutdown = true; };

      video.srcObject = stream;
      await video.play();

      captureCanvas = document.createElement('canvas');
      captureCanvas.width = 640; //video.videoWidth;
      captureCanvas.height = 480; //video.videoHeight;
      window.requestAnimationFrame(onAnimationFrame);

      return stream;
    }
    async function stream_frame(label, imgData) {
      if (shutdown) {
        removeDom();
        shutdown = false;
        return '';
      }

      var preCreate = Date.now();
      stream = await createDom();

      var preShow = Date.now();
      if (label != "") {
        labelElement.innerHTML = label;
      }

      if (imgData != "") {
        var videoRect = video.getClientRects()[0];
        imgElement.style.top = videoRect.top + "px";
        imgElement.style.left = videoRect.left + "px";
        imgElement.style.width = videoRect.width + "px";
        imgElement.style.height = videoRect.height + "px";
        imgElement.src = imgData;
      }

      var preCapture = Date.now();
      var result = await new Promise(function(resolve, reject) {
        pendingResolve = resolve;
      });
      shutdown = false;

      return {'create': preShow - preCreate,
              'show': preCapture - preShow,
              'capture': Date.now() - preCapture,
              'img': result};
    }
    ''')

  display(js)

def video_frame(label, bbox):
  data = eval_js('stream_frame("{}", "{}")'.format(label, bbox))
  return data

In [ ]:
# Load MobileNetV2 model for feature extraction
base_model = MobileNetV2(weights='imagenet', include_top=False, pooling='avg', input_shape=(224, 224, 3))
model = Model(inputs=base_model.input, outputs=base_model.output)

# Function to convert the JavaScript object into an OpenCV image
def js_to_image(js_reply):
    image_bytes = b64decode(js_reply.split(',')[1])
    jpg_as_np = np.frombuffer(image_bytes, dtype=np.uint8)
    img = cv2.imdecode(jpg_as_np, flags=1)
    return img

# Function to convert OpenCV Rectangle bounding box image into base64 byte string to overlay on video stream
def bbox_to_bytes(bbox_array):
    bbox_PIL = PIL.Image.fromarray(bbox_array, 'RGBA')
    iobuf = io.BytesIO()
    bbox_PIL.save(iobuf, format='png')
    bbox_bytes = 'data:image/png;base64,{}'.format((str(b64encode(iobuf.getvalue()), 'utf-8')))
    return bbox_bytes

# Initialize the Haar Cascade face detection model
face_cascade = cv2.CascadeClassifier(cv2.samples.findFile(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'))

# Function to sharpen the image
def sharpen_image(image, factor=1.3):
    # Sharpening kernel
    sharpening_kernel = np.array([[0, -1, 0],
                                  [-1, 4, -1],
                                  [0, -1, 0]])

    # Adjust sharpening intensity using the factor
    sharpened = cv2.filter2D(image, -1, sharpening_kernel * factor)
    return sharpened

from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Function to increase the contrast of an image by a specified factor
def increase_contrast(image, factor=1.2):
    # Convert to float32 for accurate calculations
    image = image.astype('float32')
    # Increase contrast
    image = np.clip((image - 0.5) * factor + 0.5, 0, 1)  # Scale to [0, 1]
    return (image * 255).astype(np.uint8)  # Convert back to uint8

# Initialize ImageDataGenerator for data augmentation
datagen = ImageDataGenerator(
    rotation_range=20,   # Randomly rotate images in the range (degrees, 0 to 180)
    width_shift_range=0.1,  # Randomly translate images horizontally (fraction of total width)
    height_shift_range=0.1,  # Randomly translate images vertically (fraction of total height)
    shear_range=0.2,     # Shear angle in counter-clockwise direction in degrees
    zoom_range=0.2,      # Randomly zoom image
    horizontal_flip=True, # Randomly flip images
    fill_mode='nearest'   # Fill in new pixels
)

def load_dataset_to_excel(dataset_path, output_file="embeddings.xlsx"):
    data = []  # List to store image names, labels, and embeddings

    for person_name in os.listdir(dataset_path):
        person_folder = os.path.join(dataset_path, person_name)
        if os.path.isdir(person_folder):
            for image_name in os.listdir(person_folder):
                image_path = os.path.join(person_folder, image_name)
                img = cv2.imread(image_path)

                # Resize and normalize image
                img = cv2.resize(img, (224, 224))
                img = img.astype('float32') / 255.0
                img = np.expand_dims(img, axis=0)

                # Get the embedding from the MobileNetV2 model
                embedding = model.predict(img)[0]

                # Add data to the list: [image_name, person_name, embedding]
                data.append([image_name, person_name, embedding])

    # Convert the data to a pandas DataFrame
    df = pd.DataFrame(data, columns=["Image Name", "Label", "Embedding"])

    # Split embeddings into separate columns for easier viewing in Excel
    embedding_df = pd.DataFrame(df["Embedding"].tolist())
    df = df.drop(columns=["Embedding"]).join(embedding_df)

    # Check if the output file already exists
    if os.path.exists(output_file):
        # Load existing data and append the new data
        existing_df = pd.read_excel(output_file)
        df = pd.concat([existing_df, df], ignore_index=True)

    # Save the DataFrame to the Excel file
    df.to_excel(output_file, index=False)
    print(f"Embeddings saved to {output_file}")

# Specify dataset path and output file
dataset_path = '/content/drive/MyDrive/faces'
output_file = '/content/drive/MyDrive/embeddings.xlsx'

# Run the function to save embeddings to Excel
load_dataset_to_excel(dataset_path, output_file)

In [ ]:
import tensorflow as tf
import numpy as np
import cv2
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

# Suppress TensorFlow logs
tf.get_logger().setLevel('ERROR')  # Suppress most of the output

# Function to load embeddings and labels from Excel
def load_embeddings_from_excel(excel_file):
    df = pd.read_excel(excel_file)

    # Extract labels and embeddings
    labels = df['Label'].tolist()

    # Extract embeddings as a numpy array
    embeddings = df.iloc[:, 2:].values  # Assumes embeddings start from the 3rd column
    return labels, embeddings

# Load known embeddings and labels from the saved Excel file
known_labels, known_embeddings = load_embeddings_from_excel('/content/drive/MyDrive/embeddings.xlsx')

In [ ]:

def recognise():
  # Start streaming video from webcam
  video_stream()

  # Label for video
  label_html = 'Capturing...'
  bbox = ''
  count = 0

  while True:
      js_reply = video_frame(label_html, bbox)
      if not js_reply:
          break

      img = js_to_image(js_reply["img"])

      # Create transparent overlay for bounding box
      bbox_array = np.zeros([480, 640, 4], dtype=np.uint8)

      # Grayscale image for face detection
      gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)

      # Get face region coordinates
      faces = face_cascade.detectMultiScale(gray)

      # Get face bounding box for overlay
      for (x, y, w, h) in faces:
          # Extract the face from the image
          face_roi = img[y:y+h, x:x+w]
          face_roi = cv2.resize(face_roi, (224, 224))

          # Normalize the image
          face_roi = face_roi.astype('float32') / 255.0
          face_roi = np.expand_dims(face_roi, axis=0)

          # Generate the face embedding, suppress progress bar with verbose=0
          embedding = model.predict(face_roi, verbose=0)[0]  # Disable progress bar here

          # Compare with known embeddings to find the closest match
          if len(known_embeddings) > 0:
              similarities = cosine_similarity([embedding], known_embeddings)[0]
              best_match_index = np.argmax(similarities)
              confidence = similarities[best_match_index]

              # Set a threshold for recognizing a person
              threshold = 0.84  # Adjust this threshold as needed
              if confidence > threshold:
                  name = known_labels[best_match_index]
                  label_html = f'{name}: {confidence:.2f}'
              else:
                  label_html = 'Unknown'
          else:
              label_html = 'Unknown'

          # Draw the bounding box and label on the image
          if label_html == "Unknown":
              cv2.rectangle(bbox_array, (x, y), (x + w, y + h), (255, 0, 0), 2)
              cv2.putText(bbox_array, label_html, (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2)
          else:
              cv2.rectangle(bbox_array, (x, y), (x + w, y + h), (0, 255, 0), 2)
              cv2.putText(bbox_array, label_html, (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

      # Create alpha channel for bbox_array
      bbox_array[:, :, 3] = (bbox_array.max(axis=2) > 0).astype(int) * 255
      bbox_bytes = bbox_to_bytes(bbox_array)
      bbox = bbox_bytes

recognise()


<IPython.core.display.Javascript object>

# Gradio try

In [ ]:
!pip install gradio gradio-webrtc

In [ ]:
with gr.Blocks() as demo:
    with gr.Tab("Lion"):
        gr.Image("lion.jpg")
        gr.Button("New Lion")
    with gr.Tab("Tiger"):
        gr.Image("tiger.jpg")
        gr.Button("New Tiger")
if __name__ == "__main__":
    demo.launch()

In [ ]:
def detection():
  return "hello"

In [ ]:
import gradio as gr
from gradio_webrtc import WebRTC

css = """.my-group {max-width: 600px !important; max-height: 600px !important;}
         .my-column {display: flex !important; justify-content: center !important; align-items: center !important;}"""

with gr.Blocks(css=css) as demo:
    gr.HTML(
        """
        <h1 style='text-align: center'>
        YOLOv10 Webcam Stream (Powered by WebRTC ⚡️)
        </h1>
        """
    )
    with gr.Column(elem_classes=["my-column"]):
        with gr.Group(elem_classes=["my-group"]):
            image = WebRTC(label="Stream")
            conf_threshold = gr.Slider(
                label="Confidence Threshold",
                minimum=0.0,
                maximum=1.0,
                step=0.05,
                value=0.30,
            )

        image.stream(
            fn=detection, inputs=[image, conf_threshold], outputs=[image], time_limit=10
        )

if __name__ == "__main__":
    demo.launch()

In [ ]:
import gradio as gr
from gradio_webrtc import WebRTC
import cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.applications import MobileNetV2
from sklearn.metrics.pairwise import cosine_similarity

# Load the MobileNetV2 model for feature extraction
base_model = MobileNetV2(weights='imagenet', include_top=False, pooling='avg', input_shape=(224, 224, 3))
model = Model(inputs=base_model.input, outputs=base_model.output)

# Initialize the Haar Cascade face detection model
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

# Known embeddings and labels for recognition
known_labels, known_embeddings = load_embeddings_from_excel('/content/drive/MyDrive/embeddings.xlsx')

def detection(image, conf_threshold=0.8):
    # Convert the input frame to a numpy array
    img = np.array(image)

    # Convert the frame to grayscale for face detection
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(30, 30))

    # Loop through detected faces
    for (x, y, w, h) in faces:
        # Extract and preprocess the face region for MobileNetV2
        face_roi = img[y:y+h, x:x+w]
        face_roi = cv2.resize(face_roi, (224, 224))
        face_roi = face_roi.astype('float32') / 255.0
        face_roi = np.expand_dims(face_roi, axis=0)

        # Generate the embedding for the detected face
        embedding = model.predict(face_roi, verbose=0)[0]

        # Compare with known embeddings to find the closest match
        if known_embeddings:
            similarities = cosine_similarity([embedding], known_embeddings)[0]
            best_match_index = np.argmax(similarities)
            confidence = similarities[best_match_index]

            if confidence > conf_threshold:
                name = known_labels[best_match_index]
                label = f'{name}: {confidence:.2f}'
                color = (0, 255, 0)  # Green for recognized faces
            else:
                label = 'Unknown'
                color = (255, 0, 0)  # Red for unknown faces
        else:
            label = 'Unknown'
            color = (255, 0, 0)

        # Draw bounding box and label on the frame
        cv2.rectangle(img, (x, y), (x+w, y+h), color, 2)
        cv2.putText(img, label, (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

    return img

# Define Gradio interface CSS styling
css = """.my-group {max-width: 500px !important; max-height: 500px !important;}
         .my-column {display: flex !important; justify-content: center !important; align-items: center !important;}"""

# WebRTC Configuration for streaming
rtc_configuration = {"iceServers": [{"urls": ["stun:stun.l.google.com:19302"]}]}

# Set up Gradio Blocks interface
with gr.Blocks(css=css) as demo:
    gr.HTML(
        """
        <h1 style='text-align: center'>
        Face Recognition Webcam Stream
        </h1>
        """
    )
    with gr.Column(elem_classes=["my-column"]):
        with gr.Group(elem_classes=["my-group"]):
            image = WebRTC(label="Stream", rtc_configuration=rtc_configuration)

        # Stream the function with real-time detection
        image.stream(
            fn=detection, inputs=[image], outputs=[image], time_limit=10
        )

if __name__ == "__main__":
    demo.launch()


In [ ]:
#do not ever delete it
import numpy as np
import cv2
from tensorflow.keras.models import Model
from tensorflow.keras.applications import MobileNetV2
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
import gradio as gr

# Load MobileNetV2 model for feature extraction
base_model = MobileNetV2(weights='imagenet', include_top=False, pooling='avg', input_shape=(224, 224, 3))
model = Model(inputs=base_model.input, outputs=base_model.output)

# Load face cascade for face detection
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

# Load known embeddings and labels from the Excel file
def load_embeddings_from_excel(excel_file):
    df = pd.read_excel(excel_file)
    labels = df['Label'].tolist()
    embeddings = df.iloc[:, 2:].values  # Extract embeddings as a numpy array
    return labels, embeddings

# Load embeddings and labels
known_labels, known_embeddings = load_embeddings_from_excel('/content/drive/MyDrive/embeddings.xlsx')

# Function to process each frame from the webcam
def process_frame(frame):
    # Convert to BGR (Gradio uses RGB by default)
    image = cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)

    # Convert to grayscale for face detection
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(30, 30))

    if len(faces) == 0:
        return image  # Return original image if no faces are detected

    # Process each detected face
    for (x, y, w, h) in faces:
        # Extract face ROI and preprocess for model
        face_roi = image[y:y+h, x:x+w]
        face_roi = cv2.resize(face_roi, (224, 224))
        face_roi = face_roi.astype('float32') / 255.0
        face_roi = np.expand_dims(face_roi, axis=0)

        # Perform feature extraction and similarity check
        if len(known_embeddings) > 0:
            # Run model prediction
            embedding = model.predict(face_roi, verbose=0)[0]

            # Calculate cosine similarities with known embeddings
            similarities = cosine_similarity([embedding], known_embeddings)[0]
            best_match_index = np.argmax(similarities)
            confidence = similarities[best_match_index]

            threshold = 0.80
            if confidence > threshold:
                label = known_labels[best_match_index]
                color = (0, 255, 0)  # Green for known
            else:
                label = "Unknown"
                color = (0, 0, 255)  # Red for unknown
        else:
            label = "Unknown"
            color = (0, 0, 255)

        # Draw the label and bounding box on the frame
        cv2.rectangle(image, (x, y), (x + w, y + h), color, 2)
        cv2.putText(image, f"{label}: {confidence:.2f}", (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
        # Convert to RGB before returning
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    return image

# Define Gradio interface
css = """.my-group {max-width: 600px !important; max-height: 600px !important;}
         .my-column {display: flex !important; justify-content: center !important; align-items: center !important;}"""

with gr.Blocks(css=css) as demo:
    with gr.Column(elem_classes=["my-column"]):
        with gr.Group(elem_classes=["my-group"]):
            input_img = gr.Image(sources=["webcam"], type="numpy", streaming=True, label="Input")
            #output_img = gr.Image(type="numpy", label="Output")

    # Define the function to process the image and set streaming interval
    input_img.stream(process_frame, inputs=input_img, outputs=input_img, time_limit=30, stream_every=0.1)

# Launch the interface
demo.launch()

In [ ]:
import gradio as gr
import numpy as np
import cv2
from sklearn.metrics.pairwise import cosine_similarity
from tensorflow.keras.models import Model
from tensorflow.keras.applications import MobileNetV2
from base64 import b64decode, b64encode
import pandas as pd

# Load MobileNetV2 model for feature extraction
base_model = MobileNetV2(weights='imagenet', include_top=False, pooling='avg', input_shape=(224, 224, 3))
model = Model(inputs=base_model.input, outputs=base_model.output)

# Load face cascade for face detection
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

# Load known embeddings and labels from the Excel file
def load_embeddings_from_excel(excel_file):
    df = pd.read_excel(excel_file)
    labels = df['Label'].tolist()
    embeddings = df.iloc[:, 2:].values  # Extract embeddings as a numpy array
    return labels, embeddings

known_labels, known_embeddings = load_embeddings_from_excel('/content/drive/MyDrive/embeddings.xlsx')

# Perform transformation and recognition on the frame
def process_frame(frame, transform="face_recognition"):
    if transform == "cartoon":
        img_color = cv2.pyrDown(cv2.pyrDown(frame))
        for _ in range(6):
            img_color = cv2.bilateralFilter(img_color, 9, 9, 7)
        img_color = cv2.pyrUp(cv2.pyrUp(img_color))
        img_edges = cv2.cvtColor(frame, cv2.COLOR_RGB2GRAY)
        img_edges = cv2.adaptiveThreshold(
            cv2.medianBlur(img_edges, 7),
            255,
            cv2.ADAPTIVE_THRESH_MEAN_C,
            cv2.THRESH_BINARY,
            9,
            2,
        )
        img_edges = cv2.cvtColor(img_edges, cv2.COLOR_GRAY2RGB)
        img = cv2.bitwise_and(img_color, img_edges)
        return img
    elif transform == "edges":
        img = cv2.cvtColor(cv2.Canny(frame, 100, 200), cv2.COLOR_GRAY2BGR)
        return img
    elif transform == "face_recognition":
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        faces = face_cascade.detectMultiScale(gray, 1.1, 4)

        for (x, y, w, h) in faces:
            face_roi = frame[y:y+h, x:x+w]
            face_roi = cv2.resize(face_roi, (224, 224))
            face_roi = face_roi.astype('float32') / 255.0
            face_roi = np.expand_dims(face_roi, axis=0)
            embedding = model.predict(face_roi, verbose=0)[0]

            if len(known_embeddings) > 0:
                similarities = cosine_similarity([embedding], known_embeddings)[0]
                best_match_index = np.argmax(similarities)
                confidence = similarities[best_match_index]
                threshold = 0.84

                if confidence > threshold:
                    label = known_labels[best_match_index]
                    cv2.putText(frame, f"{label}: {confidence:.2f}", (x, y - 10),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
                    color = (0, 255, 0)
                else:
                    label = "Unknown"
                    color = (0, 0, 255)
            else:
                label = "Unknown"
                color = (0, 0, 255)

            cv2.rectangle(frame, (x, y), (x + w, y + h), color, 2)
            cv2.putText(frame, label, (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

        return frame
    else:
        return frame

# Define Gradio interface
css = """.my-group {max-width: 500px !important; max-height: 500px !important;}
         .my-column {display: flex !important; justify-content: center !important; align-items: center !important};"""

with gr.Blocks(css=css) as demo:
    with gr.Column(elem_classes=["my-column"]):
        with gr.Group(elem_classes=["my-group"]):
            transform = gr.Dropdown(choices=["face_recognition", "cartoon", "edges"],
                                    value="face_recognition", label="Transformation")
            input_img = gr.Image(sources="webcam", type="numpy", streaming=True)
            output_img = gr.Image(type="numpy", label="Output")

    # Define the function to process the image and set streaming interval
    input_img.stream(process_frame, [input_img, transform], output_img, time_limit=30, stream_every=0.1)

demo.launch()


